In [1]:
## we have classifified the rep-seq.fasta and results are saved in final_taxonomy.tsv (observed taxonomies) 
# and we have a trueish taxonomy txt file as a ground truth (expected taxonomies)
# evaluate results with per sequence precision and outcome is detailed metrices for each comparisons
# metrices are: taxon detection rate, taxon accuracy rate, precision, recall and f-score.
## finally some prelim plottings to visualize results

import os
import sys
import random
import shutil
import glob
from joblib import Parallel, delayed
from os.path import join, basename, exists, expandvars, dirname, splitext, isfile
from pathlib import Path
from collections import defaultdict
import itertools
import numpy as np
import pandas as pd
from seaborn import xkcd_rgb as colors
import src.evaluate_results


In [2]:
## set path vars
n_cpus = os.cpu_count() - 1
n_jobs = 1 # number_of_jobs to run in parallel
analysis_name = "mock-community"
project_dir = expandvars("/home/kardokhk/taxonova_bench")
conda_dir = join(project_dir, "conda")
scripts_dir = join(project_dir, f"scripts/mock-community/src")
reference_database_dir = join(project_dir, f"benchmark/data/ref_dbs")
mock_dir = join(project_dir, f"benchmark/data", analysis_name)
results_dir = join(mock_dir, "results")

## some params
force = True # force run evaluations even if exists
taxa_level_ranges = range(2, 7) # 2 is class and 6 is species
seq_min_count = 1 # filter out ASV from biom is <1 hit
sample_weight = True # should the evaluations be done weighted by scale of abundance (more weighatage on abundant features)
per_seq_precision = True # evaluate Precision/Recall/Fscore on individual representative sequences

In [3]:
## name of the mock communities
mock_names = ['gutmock', 'vaginalmock', 'mockrobiota-16S', 'mockrobiota-ITS']

## amplicon regions
v_regions = ['V4', 'V3-V4', 'ITS1', 'ITS2', 'V1-V9', 'ITS12']

## mock to v_region mapping
mock_v_regions = {
    'gutmock': ['V4', 'V3-V4', 'V1-V9'],
    'vaginalmock': ['V4', 'V3-V4', 'V1-V9'],
    'mockrobiota-16S': ['V4', 'V3-V4', 'V1-V9'],
    'mockrobiota-ITS': ['ITS1', 'ITS2', 'ITS12']
}

## prepare combinations of methods and params to run for evaluations
method_parameters_combinations = {
    'taxonova': {'species_threshold': [0.5, 0.6],
                 'genus_threshold': [0.5, 0.6],
                 'family_threshold': [0.3, 0.4],
                 'order_threshold': [0.2],
                 'class_threshold': [0.1],
                 'phylum_threshold': [0.1],
                 'domain_threshold': [0.1]
                },
    'vsearch': {'p-maxaccepts': [1, 10, 100],
                'p-perc-identity': [0.80, 0.90, 0.97, 0.99],
                'p-min-consensus': [0.51, 0.75, 0.99]
                },
    'blast': {'p-maxaccepts': [1, 10, 100],
                'p-perc-identity': [0.80, 0.90, 0.97, 0.99],
                'p-min-consensus': [0.51, 0.75, 0.99]
                },
    'naive-bayes': {'p-confidence': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 
                                      0.6, 0.7, 0.8, 0.9, 1.0]
                },                                        
}

## try with first params from each
#method_parameters_combinations = {
#    tool: {param: values[0] for param, values in params.items()}
#    for tool, params in method_parameters_combinations.items()
#}

In [4]:
## generate list of expected taxonomy filepaths
expected_taxonomy_filepaths = []
for v_region in v_regions:
    for mock_name in mock_names:
        # skip if region to mock mismatch
        if v_region not in mock_v_regions[mock_name]:
            continue
        
        table_fp = join(
            mock_dir,
            mock_name,
            v_region,
            'trueish-taxonomies.tsv'
        )

        expected_taxonomy_filepaths.append(
            (mock_name, v_region, table_fp)
        )

# Convert to dict for lookup
expected_taxonomy_filepaths_lookup = {(mock_name, v_region): file_path for mock_name, v_region, file_path in expected_taxonomy_filepaths}


expected_taxonomy_filepaths

[('gutmock',
  'V4',
  '/home/kardokhk/taxonova_bench/benchmark/data/mock-community/gutmock/V4/trueish-taxonomies.tsv'),
 ('vaginalmock',
  'V4',
  '/home/kardokhk/taxonova_bench/benchmark/data/mock-community/vaginalmock/V4/trueish-taxonomies.tsv'),
 ('mockrobiota-16S',
  'V4',
  '/home/kardokhk/taxonova_bench/benchmark/data/mock-community/mockrobiota-16S/V4/trueish-taxonomies.tsv'),
 ('gutmock',
  'V3-V4',
  '/home/kardokhk/taxonova_bench/benchmark/data/mock-community/gutmock/V3-V4/trueish-taxonomies.tsv'),
 ('vaginalmock',
  'V3-V4',
  '/home/kardokhk/taxonova_bench/benchmark/data/mock-community/vaginalmock/V3-V4/trueish-taxonomies.tsv'),
 ('mockrobiota-16S',
  'V3-V4',
  '/home/kardokhk/taxonova_bench/benchmark/data/mock-community/mockrobiota-16S/V3-V4/trueish-taxonomies.tsv'),
 ('mockrobiota-ITS',
  'ITS1',
  '/home/kardokhk/taxonova_bench/benchmark/data/mock-community/mockrobiota-ITS/ITS1/trueish-taxonomies.tsv'),
 ('mockrobiota-ITS',
  'ITS2',
  '/home/kardokhk/taxonova_bench/ben

In [5]:
## generate list of observed taxonomy filepaths
observed_taxonomy_filepaths = []
for v_region in v_regions:
    for mock_name in mock_names:
        # skip if region to mock mismatch
        if v_region not in mock_v_regions[mock_name]:
            continue
        
        for method_id, param_grid in method_parameters_combinations.items():
            param_names = list(param_grid.keys())
            param_values = [v if isinstance(v, list) else [v] for v in param_grid.values()]
            for combo in itertools.product(*param_values):
                param_id = ":".join(str(v) for v in combo)

                # Generate the observed table path
                table_fp = join(
                    mock_dir,
                    mock_name,
                    v_region,
                    'classify',
                    method_id,
                    param_id,
                    'taxonomy_final.tsv'
                )

                observed_taxonomy_filepaths.append(
                    (mock_name, v_region, method_id, param_id, table_fp)
                )

observed_taxonomy_filepaths

[('gutmock',
  'V4',
  'taxonova',
  '0.5:0.5:0.3:0.2:0.1:0.1:0.1',
  '/home/kardokhk/taxonova_bench/benchmark/data/mock-community/gutmock/V4/classify/taxonova/0.5:0.5:0.3:0.2:0.1:0.1:0.1/taxonomy_final.tsv'),
 ('gutmock',
  'V4',
  'taxonova',
  '0.5:0.5:0.4:0.2:0.1:0.1:0.1',
  '/home/kardokhk/taxonova_bench/benchmark/data/mock-community/gutmock/V4/classify/taxonova/0.5:0.5:0.4:0.2:0.1:0.1:0.1/taxonomy_final.tsv'),
 ('gutmock',
  'V4',
  'taxonova',
  '0.5:0.6:0.3:0.2:0.1:0.1:0.1',
  '/home/kardokhk/taxonova_bench/benchmark/data/mock-community/gutmock/V4/classify/taxonova/0.5:0.6:0.3:0.2:0.1:0.1:0.1/taxonomy_final.tsv'),
 ('gutmock',
  'V4',
  'taxonova',
  '0.5:0.6:0.4:0.2:0.1:0.1:0.1',
  '/home/kardokhk/taxonova_bench/benchmark/data/mock-community/gutmock/V4/classify/taxonova/0.5:0.6:0.4:0.2:0.1:0.1:0.1/taxonomy_final.tsv'),
 ('gutmock',
  'V4',
  'taxonova',
  '0.6:0.5:0.3:0.2:0.1:0.1:0.1',
  '/home/kardokhk/taxonova_bench/benchmark/data/mock-community/gutmock/V4/classify/taxonova/

In [6]:
## generate expected_composition biom with metadata information
for v_region in v_regions:
    for mock_name in mock_names:
        # skip if region to mock mismatch
        if v_region not in mock_v_regions[mock_name]:
            continue
        
        ## prepare expected biom table with metadata information on taxonomic lineage
        exp_table_fp = join(mock_dir, mock_name, v_region, 'expected_composition.biom')
        if not exists(exp_table_fp):
            src.evaluate_results.add_metadata_to_biom_table(
                input_fp = join(mock_dir, mock_name, v_region, 'env_abundance.tsv'),
                taxonomy_map_fp = None,
                biom_output_fp = exp_table_fp,
                mode='expected'
            )

In [7]:
## generate observed_composition biom with metadata information
for mock_id, region_id, method_id, param_id, obs_taxa_fp in observed_taxonomy_filepaths:

    ## if classifications yet to be run for this combinations
    if not exists(obs_taxa_fp):
        continue

    ## add classification results as a metadata to obs_table_fp and save for each classification combinations
    obs_table_fp = join(os.path.dirname(obs_taxa_fp), 'observed_composition.biom')
    if not exists(obs_table_fp):
        src.evaluate_results.add_metadata_to_biom_table(
            input_fp = glob.glob(join(mock_dir, mock_id, region_id, 'source', '*', f'table_{mock_id}.biom'))[0],
            taxonomy_map_fp = obs_taxa_fp,
            biom_output_fp = obs_table_fp,
            mode='observed'
        )

In [8]:
## taxa level map
taxonomy_level_map = {0:"Domain", 1:"Phylum", 2:"Class", 3:"Order", 4:"Family", 5:"Genus", 6:"Species"}

## compare observed and expected result tables
all_results = []
for mock_id, region_id, method_id, param_id, obs_table_fp in observed_taxonomy_filepaths:

    ## if classifications yet to be run for this combinations
    if not exists(obs_table_fp):
        continue

    # get expected and observed taxonomies
    obs_taxa_fp = obs_table_fp
    exp_taxa_fp = expected_taxonomy_filepaths_lookup.get((mock_id, region_id))

    ## observed biom file
    obs_table_fp = join(os.path.dirname(obs_taxa_fp), 'observed_composition.biom')
    feature_table = obs_table_fp # just to access and match observations (i.e. rep_seq ids)

    ## expected biom file
    exp_table_fp = join(mock_dir, mock_id, region_id, 'expected_composition.biom')

    # evaluate results for each taxonomic level
    for taxa_level in taxa_level_ranges:
        outfile = join(os.path.dirname(obs_taxa_fp), 'evaluation.txt')
        if not exists(outfile) or force:
            out = src.evaluate_results.compute_results(
                expected_table_fp = exp_table_fp, 
                observed_table_fp = obs_table_fp, 
                feature_table_fp = feature_table,
                expected_taxa_fp = exp_taxa_fp, 
                observed_taxa_fp = obs_taxa_fp,
                taxonomy_level = taxa_level,
                metadata = (mock_id, region_id, method_id, param_id, taxonomy_level_map[taxa_level]),
                out_file = outfile,
                min_count = seq_min_count,
                sample_weight = sample_weight,
                per_seq_precision=True,
                md_key='taxonomy',
                taxa_to_keep=None,
                exclude=None)
            # append to list
            all_results.append(out)
        else:
            out = pd.read_csv(outfile, sep="\t", index_col=None)
            

# Combine all into one final dataframe
final_result = pd.concat(all_results, ignore_index=True)


In [9]:
final_result

,mock_name,v_region,method,param,taxa_level,sample_id,n_exp_taxa,n_obs_taxa,tp,fp,fn,Taxon Accuracy Rate,Taxon Detection Rate,n_exp_seq,n_obs_seq,Precision,Recall,F-measure,per_seq_ids,per_seq_matches
0,gutmock,V4,taxonova,0.5:0.5:0.3:0.2:0.1:0.1:0.1,Class,gutmock00-1,1,1,1,0,0,1.0,1.0,74,74,1.0,1.0,1.0,"81b14a06fe51fed6aaa311ff4e370125,385197fd9f310...","match,match,match,match,match,match,match,matc..."
1,gutmock,V4,taxonova,0.5:0.5:0.3:0.2:0.1:0.1:0.1,Class,gutmock00-2,1,1,1,0,0,1.0,1.0,76,76,1.0,1.0,1.0,"81b14a06fe51fed6aaa311ff4e370125,385197fd9f310...","match,match,match,match,match,match,match,matc..."
2,gutmock,V4,taxonova,0.5:0.5:0.3:0.2:0.1:0.1:0.1,Class,gutmock00-3,1,1,1,0,0,1.0,1.0,73,73,1.0,1.0,1.0,"81b14a06fe51fed6aaa311ff4e370125,385197fd9f310...","match,match,match,match,match,match,match,matc..."
3,gutmock,V4,taxonova,0.5:0.5:0.3:0.2:0.1:0.1:0.1,Class,gutmock01-1,1,1,1,0,0,1.0,1.0,77,77,1.0,1.0,1.0,"81b14a06fe51fed6aaa311ff4e370125,385197fd9f310...","match,match,match,match,match,match,match,matc..."
4,gutmock,V4,taxonova,0.5:0.5:0.3:0.2:0.1:0.1:0.1,Class,gutmock01-2,1,1,1,0,0,1.0,1.0,77,77,1.0,1.0,1.0,"81b14a06fe51fed6aaa311ff4e370125,385197fd9f310...","match,match,match,match,match,match,match,matc..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163795,mockrobiota-ITS,ITS12,naive-bayes,1.0,Species,mockrobiota-ITS08-2,10,7,0,7,10,0.0,0.0,14,14,0.0,0.0,0.0,"24e47f5e05f361f70933d76a207e447b,fe54e781b7da2...","underclassification,misclassification,misclass..."
163796,mockrobiota-ITS,ITS12,naive-bayes,1.0,Species,mockrobiota-ITS08-3,10,7,0,7,10,0.0,0.0,14,14,0.0,0.0,0.0,"24e47f5e05f361f70933d76a207e447b,fe54e781b7da2...","underclassification,misclassification,misclass..."
163797,mockrobiota-ITS,ITS12,naive-bayes,1.0,Species,mockrobiota-ITS09-1,9,6,0,6,9,0.0,0.0,13,13,0.0,0.0,0.0,"24e47f5e05f361f70933d76a207e447b,c9772c8b28517...","underclassification,misclassification,misclass..."
163798,mockrobiota-ITS,ITS12,naive-bayes,1.0,Species,mockrobiota-ITS09-2,9,6,0,6,9,0.0,0.0,13,13,0.0,0.0,0.0,"24e47f5e05f361f70933d76a207e447b,c9772c8b28517...","underclassification,misclassification,misclass..."


In [10]:
## Export final_result table as TSV file
# Save the final_result table to the same directory where the evaluation plots are saved
tsv_output_path = join(results_dir, "final_result_evaluation.tsv")

# Create results directory if it doesn't exist
if not exists(results_dir):
    os.makedirs(results_dir)

# Export the table as TSV
final_result.to_csv(tsv_output_path, sep='\t', index=False)
print(f"Final result table exported to: {tsv_output_path}")

Final result table exported to: /home/kardokhk/taxonova_bench/benchmark/data/mock-community/results/final_result_evaluation.tsv


In [11]:
from importlib import reload
reload(src.evaluate_results)

<module 'src.evaluate_results' from '/home/kardokhk/taxonova_bench/scripts/mock-community/src/evaluate_results.py'>

In [12]:
## Quick Plot
## params
only_plot_taxonova = False
plot_by_param = False
plot_by_region = True  # Add new parameter for plotting by region
plot_by_mock = False   # Add new parameter for plotting by mock community
if only_plot_taxonova: 
    plot_by_param = True

# Set line_by based on plot_by_mock flag
if plot_by_mock:
    line_by = "mock_name"
else:
    line_by = None
#line_by = None

## outdir
out_dir = results_dir
if not exists(out_dir):
    os.makedirs(out_dir)

## reset index
mock_results = final_result.reset_index(drop=True)

## color code
color_palette = {
    'rdp': colors['baby shit green'], 'sortmerna': colors['macaroni and cheese'],
    'uclust': 'coral', 'blast': colors['electric purple'], 'naive-bayes': 'dodgerblue',
    'naive-bayes-bespoke': 'blue', 'vsearch': 'firebrick', 'taxonova': 'black'
}

## lines code
line_dict = {
    "mockrobiota-16S": "", # solid
    "mockrobiota-ITS": (1, 1), # dotted
    "gutmock": (2, 2), # dashed
    "vaginalmock": (3, 2, 1, 2), # dash-dot pattern
    "skinmock": (5, 2), # long dash
}


## evaluation metrices to plot
y_vars = ["Precision", "Recall", "F-measure", "Taxon Accuracy Rate", "Taxon Detection Rate"]

## only plot results for taxonova if requested
if only_plot_taxonova:
    mock_results = mock_results[mock_results['method'] == "taxonova"]

## plot results
if not plot_by_param and not plot_by_region and not plot_by_mock:
    # plot results from all params, regions, and mocks combined
    src.evaluate_results.plot_from_data_frame(
        df=mock_results,
        outdir=out_dir,
        x_axis="taxa_level",
        y_vars=y_vars,
        color_by="method",
        color_palette=color_palette,
        line_by=line_by
    )

elif plot_by_mock:
    # plot all mock communities separately with organized directories
    unique_mocks = mock_results['mock_name'].unique()
    
    for mock in unique_mocks:
        # Create directory for each mock community
        mock_dir_path = join(out_dir, f"by_mock/{mock}")
        if not exists(mock_dir_path):
            os.makedirs(mock_dir_path)
        
        mock_subset = mock_results[mock_results['mock_name'] == mock]
        
        if plot_by_region:
            # plot by region within each mock
            unique_regions = mock_subset['v_region'].unique()
            
            for region in unique_regions:
                region_subset = mock_subset[mock_subset['v_region'] == region]
                
                if plot_by_param:
                    # plot by param within each region and mock
                    unique_params = region_subset['param'].unique()
                    
                    for param in unique_params:
                        param_subset = region_subset[region_subset['param'] == param]
                        
                        src.evaluate_results.plot_from_data_frame(
                            df=param_subset,
                            outdir=mock_dir_path,
                            x_axis="taxa_level",
                            y_vars=y_vars,
                            color_by="method",
                            color_palette=color_palette,
                            line_by=None,  # No line_by when plotting by mock separately
                            out_file=f"{mock}_{region}_{param}_eval_results.pdf"
                        )
                else:
                    # plot all params combined for each region and mock
                    src.evaluate_results.plot_from_data_frame(
                        df=region_subset,
                        outdir=mock_dir_path,
                        x_axis="taxa_level",
                        y_vars=y_vars,
                        color_by="method",
                        color_palette=color_palette,
                        line_by=None,  # No line_by when plotting by mock separately
                        out_file=f"{mock}_{region}_eval_results.pdf"
                    )
        elif plot_by_param:
            # plot by param within each mock
            unique_params = mock_subset['param'].unique()
            
            for param in unique_params:
                param_subset = mock_subset[mock_subset['param'] == param]
                
                src.evaluate_results.plot_from_data_frame(
                    df=param_subset,
                    outdir=mock_dir_path,
                    x_axis="taxa_level",
                    y_vars=y_vars,
                    color_by="method",
                    color_palette=color_palette,
                    line_by=None,  # No line_by when plotting by mock separately
                    out_file=f"{mock}_{param}_eval_results.pdf"
                )
        else:
            # plot all params and regions combined for each mock
            src.evaluate_results.plot_from_data_frame(
                df=mock_subset,
                outdir=mock_dir_path,
                x_axis="taxa_level",
                y_vars=y_vars,
                color_by="method",
                color_palette=color_palette,
                line_by=None,  # No line_by when plotting by mock separately
                out_file=f"{mock}_eval_results.pdf"
            )

elif plot_by_region:
    # plot all regions separately with organized directories
    unique_regions = mock_results['v_region'].unique()
    
    for region in unique_regions:
        # Create directory for each region
        region_dir = join(out_dir, f"by_region/{region}")
        if not exists(region_dir):
            os.makedirs(region_dir)
        
        region_subset = mock_results[mock_results['v_region'] == region]
        
        if plot_by_param:
            # plot by param within each region
            unique_params = region_subset['param'].unique()
            
            for param in unique_params:
                param_subset = region_subset[region_subset['param'] == param]
                
                src.evaluate_results.plot_from_data_frame(
                    df=param_subset,
                    outdir=region_dir,
                    x_axis="taxa_level",
                    y_vars=y_vars,
                    color_by="method",
                    color_palette=color_palette,
                    line_by=line_by,
                    out_file=f"{region}_{param}_eval_results.pdf"
                )
        else:
            # plot all params combined for each region
            src.evaluate_results.plot_from_data_frame(
                df=region_subset,
                outdir=region_dir,
                x_axis="taxa_level",
                y_vars=y_vars,
                color_by="method",
                color_palette=color_palette,
                line_by=line_by,
                out_file=f"{region}_eval_results.pdf"
            )

elif plot_by_param:
    # plot all params separately (original functionality)
    unique_params = mock_results['param'].unique()

    for param in unique_params:
        subset_df = mock_results[mock_results['param'] == param]

        src.evaluate_results.plot_from_data_frame(
            df=subset_df,
            outdir=out_dir,
            x_axis="taxa_level",
            y_vars=y_vars,
            color_by="method",
            color_palette=color_palette,
            line_by=line_by,
            out_file=f"{param}_eval_results.pdf"
        )